## Baseline: Logistic Regression

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from data import load_gahd
from sklearn.metrics import f1_score, precision_score, recall_score

df_train, df_dev, df_test = load_gahd()

print(len(df_train), len(df_dev), len(df_test))

baseline = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5)),
    LogisticRegression(max_iter=1000),
)
baseline.fit(df_train["text"], df_train["label"])
preds = baseline.predict(df_test["text"])


y = df_test["label"]
metrics = {
    "f1_macro": f1_score(y, preds, average="macro"),
    "f1_hate": f1_score(y, preds, pos_label=1),
    "f1_not_hate": f1_score(y, preds, pos_label=0),
    "precision_hate": precision_score(y, preds),
    "recall_hate": recall_score(y, preds),
}
print({k: round(v, 4) for k, v in metrics.items()})




7701 1649 1646


{'f1_macro': 0.6926, 'f1_hate': 0.6166, 'f1_not_hate': 0.7686, 'precision_hate': 0.6984, 'recall_hate': 0.552}


## Visualizations

Diagnostics for the TF-IDF baseline on the GAHD test set. The plotting code lives in `visualizations.py`; hover over any chart for exact values.

In [2]:
import pandas as pd

import visualizations as viz

viz.apply_style()
test_results = viz.with_predictions(df_test, baseline.decision_function(df_test["text"]))

### Confusion matrix

Rows are normalised by ground truth, so the diagonal reads as per-class recall.

In [3]:
viz.plot_confusion_matrix(test_results)

### Precision-recall curve

The dot marks the 0.5 threshold that produced the metrics above. The dashed line is the hate share, which is what a random classifier scores.

In [4]:
viz.plot_precision_recall(test_results, "baseline")

### Most indicative words

Each word that occurs at least five times in training is classified on its own, and the bars show its log-odds of hate. The scores show what the model learned, not whether a word is hateful: a neutral identity term like *frauen* scores as hate because most training sentences containing it are labelled hate. Hover over a bar for those counts.

In [5]:
words = viz.frequent_words(df_train["text"])
viz.plot_word_scores(pd.Series(baseline.decision_function(words), index=words), df_train)